# NiyamTrace-X — OpenAI + Gemini + Groq Provider Preflight

This notebook asks for **three API keys securely** using hidden input:

- OpenAI API key
- Gemini API key
- Groq API key

No key is written to disk.

It verifies:
1. provider connection,
2. model availability,
3. ordinary chat,
4. native function/tool calling,
5. which **independent model families** are usable for the final closure experiments.

If OpenAI API quota is unavailable, the final closure notebook can still use:
- Gemini,
- Groq Qwen,
- Groq Llama or Groq GPT-OSS.

> If an API key was pasted into a chat or public place, rotate/revoke it before using this notebook.

In [ ]:
# CELL 1 — INSTALL/IMPORT CLIENT
import sys, subprocess, os, json, pandas as pd
from getpass import getpass

try:
    from openai import OpenAI
except Exception:
    p=subprocess.run([sys.executable,"-m","pip","install","-q","-U","openai"],capture_output=True,text=True)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError("Could not install openai client.")
    from openai import OpenAI

print("Client ready.")


In [ ]:
# CELL 2 — SECURELY ASK FOR ALL THREE KEYS
def ask_secret(env_name, prompt):
    v=os.getenv(env_name,"").strip()
    if not v:
        v=getpass(prompt+": ").strip()
    if v:
        os.environ[env_name]=v
    return v

OPENAI_API_KEY=ask_secret("OPENAI_API_KEY","OpenAI API key (hidden; Enter to skip)")
GEMINI_API_KEY=ask_secret("GEMINI_API_KEY","Gemini API key (hidden; Enter to skip)")
GROQ_API_KEY=ask_secret("GROQ_API_KEY","Groq API key (hidden; Enter to skip)")

print("OpenAI key:", "SET" if OPENAI_API_KEY else "SKIPPED")
print("Gemini key:", "SET" if GEMINI_API_KEY else "SKIPPED")
print("Groq key:", "SET" if GROQ_API_KEY else "SKIPPED")

In [ ]:
# CELL 3 — PROVIDER + MODEL CANDIDATES
PROVIDERS={
    "openai":{
        "api_key":OPENAI_API_KEY,
        "base_url":"https://api.openai.com/v1",
        "candidates":[
            {"family":"OpenAI-GPT","model":"gpt-5.6-luna"},
            {"family":"OpenAI-GPT","model":"gpt-5.6-terra"},
        ],
    },
    "gemini":{
        "api_key":GEMINI_API_KEY,
        "base_url":"https://generativelanguage.googleapis.com/v1beta/openai/",
        "candidates":[
            {"family":"Gemini","model":"gemini-3.8-flash"},
        ],
    },
    "groq":{
        "api_key":GROQ_API_KEY,
        "base_url":"https://api.groq.com/openai/v1",
        "candidates":[
            {"family":"Qwen","model":"qwen/qwen3.6-27b"},
            {"family":"GPT-OSS","model":"openai/gpt-oss-20b"},
            {"family":"Llama","model":"llama-3.1-8b-instant"},
            {"family":"Llama","model":"llama-3.3-70b-versatile"},
        ],
    },
}

In [ ]:
# CELL 4 — LIST MODELS FROM EACH PROVIDER
catalog_rows=[]
catalogs={}

for provider,cfg in PROVIDERS.items():
    if not cfg["api_key"]:
        catalogs[provider]=set()
        continue
    try:
        client=OpenAI(api_key=cfg["api_key"],base_url=cfg["base_url"])
        models=client.models.list()
        ids=set()
        for m in models:
            mid=getattr(m,"id",None)
            if mid: ids.add(str(mid))
        catalogs[provider]=ids
        catalog_rows.append({"provider":provider,"status":"OK","models_found":len(ids),"error":""})
    except Exception as e:
        catalogs[provider]=set()
        catalog_rows.append({"provider":provider,"status":"ERROR","models_found":0,"error":repr(e)})

display(pd.DataFrame(catalog_rows))

In [ ]:
# CELL 5 — CHAT + TOOL-CALL PREFLIGHT
tool=[{
    "type":"function",
    "function":{
        "name":"lookup_order",
        "description":"Look up an order",
        "parameters":{
            "type":"object",
            "properties":{"order_id":{"type":"string"}},
            "required":["order_id"]
        }
    }
}]

rows=[]
for provider,cfg in PROVIDERS.items():
    if not cfg["api_key"]:
        continue
    client=OpenAI(api_key=cfg["api_key"],base_url=cfg["base_url"])
    available=catalogs.get(provider,set())
    for c in cfg["candidates"]:
        model=c["model"]
        # Some provider catalogs may omit aliases; still try preferred candidates.
        r={"provider":provider,"family":c["family"],"model":model}
        try:
            a=client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":"Reply exactly OK"}],
                max_tokens=16,
            )
            r["chat_ok"]=bool(a.choices)
            r["chat_error"]=""
        except Exception as e:
            r["chat_ok"]=False
            r["chat_error"]=repr(e)

        try:
            b=client.chat.completions.create(
                model=model,
                messages=[{"role":"user","content":"Use lookup_order for order A123."}],
                tools=tool,
                tool_choice="auto",
                max_tokens=96,
            )
            msg=b.choices[0].message if b.choices else None
            calls=getattr(msg,"tool_calls",None) if msg else None
            r["tool_ok"]=bool(calls)
            r["tool_error"]=""
        except Exception as e:
            r["tool_ok"]=False
            r["tool_error"]=repr(e)

        r["status"]="OK" if r["chat_ok"] and r["tool_ok"] else "FAILED"
        rows.append(r)

preflight=pd.DataFrame(rows)
display(preflight)

In [ ]:
# CELL 6 — CHOOSE THREE INDEPENDENT FAMILIES
# Prefer provider diversity first:
priority=[
    ("openai","OpenAI-GPT"),
    ("gemini","Gemini"),
    ("groq","Qwen"),
    ("groq","Llama"),
    ("groq","GPT-OSS"),
]

selected=[]
used_families=set()

for provider,family in priority:
    hit=preflight[
        (preflight.provider==provider)&
        (preflight.family==family)&
        (preflight.status=="OK")
    ]
    if len(hit) and family not in used_families:
        row=hit.iloc[0]
        selected.append({
            "provider":row.provider,
            "family":row.family,
            "model":row.model,
            "base_url":PROVIDERS[row.provider]["base_url"],
        })
        used_families.add(family)
    if len(selected)>=3:
        break

selected_df=pd.DataFrame(selected)
display(selected_df)

if len(selected)>=3:
    print("READY: three independent model families are available.")
else:
    print("NOT READY: only",len(selected),"independent families passed.")
    print("You may still proceed later if another provider/key becomes available.")

# Export only non-secret configuration.
selected_df.to_json("NTX_SELECTED_PROVIDER_MODELS.json",orient="records",indent=2)
print("Saved: NTX_SELECTED_PROVIDER_MODELS.json")